In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

In [ ]:
import torch
print("torch =", torch.__version__)
print("cuda  =", torch.version.cuda)

In [ ]:
!python -m pip uninstall -y pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric
!python -m pip install --no-cache-dir networkx
!python -m pip install --upgrade pip

In [ ]:
!python -m pip install --no-cache-dir \
  pyg-lib==0.4.0 \
  torch-scatter==2.1.2 \
  torch-sparse==0.6.18 \
  torch-cluster==1.6.3 \
  torch-spline-conv==1.2.2 \
  torch-geometric==2.6.1 \
  -f https://data.pyg.org/whl/torch-2.6.0+cu126.html

In [ ]:
!python -m pip install --no-cache-dir torch-geometric-temporal
!pip install tqdm

In [ ]:
# # optional: TF32 for faster matmuls (speed, not memory)
# if use_cuda and torch.cuda.get_device_capability()[0] >= 8:
#     torch.backends.cuda.matmul.allow_tf32 = True
#     torch.backends.cudnn.allow_tf32 = True

# def log_mem(tag=""):
#     if not use_cuda: return
#     alloc = torch.cuda.memory_allocated() / (1024**2)
#     reserv = torch.cuda.memory_reserved() / (1024**2)
#     peak  = torch.cuda.max_memory_allocated() / (1024**2)
#     print(f"[MEM] {tag} alloc={alloc:.0f}MB reserved={reserv:.0f}MB peak={peak:.0f}MB")


In [ ]:
cd ..

In [ ]:
import os
from pathlib import Path
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import from_networkx
from torch.utils.data import Dataset
from torch_geometric.loader import NeighborLoader#, DataLoader
from torch_geometric_temporal.nn.recurrent import GConvGRU, TGCN, GConvLSTM, A3TGCN2
from src.dataloader import GraphTimeSeriesDataset, GraphTimeSeriesLoader
from src.temporal_split import temporal_train_test_split
from torch.utils.data import DataLoader
from src.sparse_dataloader import TrafficDataset
from src.training_utils import pad_collate, Node50Picker, build_node_50_picker, EpochUniquePicker
from tqdm.auto import tqdm

from collections import defaultdict
from datetime import datetime
import time
import torch.amp
from torch.cuda.amp import autocast, GradScaler
import pandas as pd

In [ ]:
processed_data_dir = "../custom-file-systems/efs/fs-099e29197c0a3e353/data/datos_procesados"
sparse_data_dir = f"{processed_data_dir}/sparse_datasets"
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"
static_data_filename = f"{sparse_data_dir}/static_dataset.h5"
dynamic_data_paths = [
    f"{sparse_data_dir}/coi_points_dataset.h5",
    f"{sparse_data_dir}/coi_tracts_dataset.h5",
    f"{sparse_data_dir}/irregularities_dataset.h5",
    f"{sparse_data_dir}/events_dataset.h5",
    f"{sparse_data_dir}/incidents_dataset.h5",
    f"{sparse_data_dir}/jams_variables_dataset.h5",
]
metadata_paths = [
    f"{sparse_data_dir}/coi_points_dataset_metadata.pkl",
    f"{sparse_data_dir}/coi_tracts_dataset_metadata.pkl",
    f"{sparse_data_dir}/irregularities_dataset_metadata.pkl",
    f"{sparse_data_dir}/events_dataset_metadata.pkl",
    f"{sparse_data_dir}/incidents_dataset_metadata.pkl",
    f"{sparse_data_dir}/jams_variables_dataset_metadata.pkl",
]
target_data_filename = f"{sparse_data_dir}/jams_dataset.h5"

In [ ]:
class StaticEncoder(nn.Module):
    def __init__(self, static_dim, out_dim=32):
        super().__init__()
        self.fc = nn.Linear(static_dim, out_dim)

    def forward(self, x):  # x: (N, F_static) or (n, F_static)
        return F.relu(self.fc(x))

In [ ]:
def make_batched_edge_index(edge_index: torch.Tensor, N: int, B: int, device):
    # edge_index: (2, E)
    E = edge_index.size(1)
    ei = edge_index.repeat(1, B).to(device)            # (2, E*B)
    offsets = (torch.arange(B, device=device) * N).repeat_interleave(E)
    ei[0] += offsets
    ei[1] += offsets
    return ei  # (2, E*B)

In [ ]:
### The normal version

# === General model framework ===
# === General model framework ===
class GNN_Temporal_Model(nn.Module):
    def __init__(self, edge_index, static_dim, dynamic_dim, future_dim,
                 input_seq_len, forecast_horizon, model_type="GConvGRU"):
        super().__init__()
        self.edge_index = None
        self.static_encoder = StaticEncoder(static_dim)

        if model_type == "GConvGRU":
            self.recurrent = GConvGRU(32 + dynamic_dim, 32, K=2)
            self.future_recurrent = GConvGRU(32 + future_dim, 32, K=2)
        elif model_type == "TGCN":
            self.recurrent = TGCN(32 + dynamic_dim, 32)
            self.future_recurrent = TGCN(32 + future_dim, 32)
        elif model_type == "GConvLSTM":
            self.recurrent = GConvLSTM(32 + dynamic_dim, 32, K=2)
            self.future_recurrent = GConvLSTM(32 + future_dim, 32, K=2)
        elif model_type == "A3TGCN":
            self.recurrent = A3TGCN2(in_channels=32 + dynamic_dim,
                                     out_channels=32,
                                     periods=input_seq_len,
                                     kernel=2)
            self.future_recurrent = A3TGCN2(in_channels=32 + future_dim,
                                            out_channels=32,
                                            periods=forecast_horizon,
                                            kernel=2)
        else:
            raise ValueError("Unknown model type")

        self.linear = nn.Linear(32, 2)
        self.model_type = model_type
        
    def forward(self, edge_index, static_inputs, dynamic_seq_inputs, future_known_inputs):
        # edge_index: (2, E), static_inputs: (N, Fstat)
        # dynamic_seq_inputs: (B, Tin, N, Fdyn), future_known_inputs: (B, H, N, Ff)
        B, Tin, N, Fdyn = dynamic_seq_inputs.shape
        H = future_known_inputs.shape[1]
        device = static_inputs.device
    
        # Precompute batched graph once
        if (self.edge_index is None or
            self.edge_index.shape[1] != edge_index.shape[1] * B):
            self.edge_index = make_batched_edge_index(edge_index, N, B, device)
    
        # Static embedding, then tile across batch
        static_embed = self.static_encoder(static_inputs)        # (N, 32)
        static_embed = static_embed.unsqueeze(0).expand(B, N, 32)  # (B,N,32)
    
        # Hidden state (B,N,32) -> flatten to (B*N, 32)
        h = torch.zeros(B, N, 32, device=device)
        h_flat = h.reshape(B*N, 32)
    
        # ---- Past sequence ----
        if self.model_type != "A3TGCN":
            for t in range(Tin):
                x_t = dynamic_seq_inputs[:, t]                  # (B,N,Fdyn)
                combined = torch.cat([static_embed, x_t], -1)   # (B,N,32+Fdyn)
                x_flat = combined.reshape(B*N, 32+Fdyn)         # (B*N, in)
                h_flat = self.recurrent(x_flat, self.edge_index, H=h_flat)  # (B*N,32)
        else:
            # Build full (Tin, B*N, in) then reshape to (B*N, Tin, in) if your cell expects that
            seq = []
            for t in range(Tin):
                x_t = dynamic_seq_inputs[:, t]
                combined = torch.cat([static_embed, x_t], -1)   # (B,N,in)
                seq.append(combined.reshape(B*N, -1).unsqueeze(1))
            full_seq = torch.cat(seq, dim=1)                    # (B*N, Tin, in)
            h_flat = self.recurrent(full_seq, self.edge_index)  # -> (B*N,32) or similar
    
        # ---- Future steps ----
        outs = []
        for t in range(H):
            f_t = future_known_inputs[:, t]                     # (B,N,Ff)
            combined_f = torch.cat([h_flat.view(B, N, 32), f_t], -1)  # (B,N,32+Ff)
            x_flat = combined_f.reshape(B*N, 32 + f_t.size(-1))
            h_flat = self.future_recurrent(x_flat, self.edge_index, H=h_flat)
            out_t = self.linear(h_flat).view(B, N, 2)           # (B,N,1)
            outs.append(out_t)
    
        return torch.stack(outs, dim=2)  # (B, N, H)
    # def forward(self, edge_index, static_inputs, dynamic_seq_inputs, future_known_inputs):
    #     self.edge_index = edge_index
    #     B, T_in, N, D = dynamic_seq_inputs.shape
    #     H = future_known_inputs.shape[1]  # forecast horizon

    #     # Encode static features
    #     static_embed = self.static_encoder(static_inputs)  # [N, 32]
    #     static_embed = static_embed.unsqueeze(0).repeat(B, 1, 1)  # [B, N, 32]

    #     # Initialize hidden state
    #     h = torch.zeros(B, N, 32, device=static_inputs.device)

    #     # === Past sequence processing ===
    #     if self.model_type != "A3TGCN":
    #         for t in range(T_in):
    #             x_t = dynamic_seq_inputs[:, t]  # [B, N, D]
    #             combined = torch.cat([static_embed, x_t], dim=-1)  # [B, N, 32+D]

    #             # build a new list for h to avoid in-place modification
    #             new_h = []
    #             for b in range(B):
    #                 updated = self.recurrent(combined[b], self.edge_index, H=h[b])
    #                 new_h.append(updated)
    #             h = torch.stack(new_h, dim=0)  # [B, N, 32]
    #     else:
    #         full_seq = []
    #         for t in range(T_in):
    #             x_t = dynamic_seq_inputs[:, t]
    #             combined = torch.cat([static_embed, x_t], dim=-1)
    #             full_seq.append(combined.unsqueeze(1))
    #         full_seq = torch.cat(full_seq, dim=1)

    #         new_h = []
    #         for b in range(B):
    #             updated = self.recurrent(full_seq[b], self.edge_index)
    #             new_h.append(updated)
    #         h = torch.stack(new_h, dim=0)  # [B, N, 32]

    #     # === Future steps recurrent modeling ===
    #     outputs = []
    #     for t in range(H):
    #         f_t = future_known_inputs[:, t]  # [B, N, future_dim]
    #         combined_f = torch.cat([h, f_t], dim=-1)  # [B, N, 32+future_dim]

    #         new_h = []
    #         for b in range(B):
    #             updated = self.future_recurrent(combined_f[b], self.edge_index, H=h[b])
    #             new_h.append(updated)
    #         h = torch.stack(new_h, dim=0)  # [B, N, 32]

    #         out_t = self.linear(h)  # [B, N, 1]
    #         outputs.append(out_t)

    #     return torch.cat(outputs, dim=-1)  # [B, N, H]

In [ ]:
in_timesteps = 150
out_timesteps = 50

In [ ]:
traffic_dataset = TrafficDataset(
    graph_path=graph_filename,
    static_data_path=static_data_filename,
    dynamic_data_paths=dynamic_data_paths,
    metadata_paths=metadata_paths, 
    target_data_path=target_data_filename,
    in_timesteps=in_timesteps,
    out_timesteps=out_timesteps,
    future_features=[
        "coi_puntos",  # ajusta a las columnas reales disponibles
        "coi_tramo",
        "AGLOMERACION",
        "EMERGENCIA",
        "MANIFESTACION",
    ],
    node_batch_size=500,  # <- carga todos los nodos en cada sample
    weather_data_dir=sparse_data_dir,
    include_weather=True,
    neighbor_features_path=f"{processed_data_dir}/neighbor_features.parquet",
    normalization_constants_path=f"{sparse_data_dir}/normalization_constants.pkl",
    include_velocity_target=True
)

In [ ]:
train_dataset, test_dataset = temporal_train_test_split(traffic_dataset, train_ratio=0.8)

#### 

In [ ]:
batch_size = 16
traffic_dataloader = DataLoader(
    train_dataset,           # created with node_batch_size=... (your code supports this) :contentReference[oaicite:3]{index=3}
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=pad_collate,
)

In [ ]:
base_picker = build_node_50_picker("data/datos_procesados/nodes_traffic_info.csv", top_frac=0.2, seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
static_dim = 117
dynamic_dim = 126
future_dim = 5


In [ ]:
model = GNN_Temporal_Model(
    edge_index=None,                 # will be set on forward call
    static_dim=static_dim,
    dynamic_dim=dynamic_dim,
    future_dim=future_dim,
    input_seq_len=in_timesteps,
    forecast_horizon=out_timesteps,
    model_type="GConvGRU",
).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [ ]:
assert getattr(traffic_dataloader, "num_workers", 0) == 0, \
    "For per-batch seed changes, set DataLoader(num_workers=0) or use a BatchSampler design."

In [ ]:
model.train()

In [ ]:
num_epochs = 2

In [ ]:
def _now(): return datetime.now().strftime('%H:%M:%S.%f')[:-3]
def _log(msg): tqdm.write(f"[{_now()}] {msg}")

def stats(name, t):
    # Use on reasonably sized tensors (or summarize before calling)
    n_nan = torch.isnan(t).sum().item()
    n_inf = torch.isinf(t).sum().item()
    maxabs = torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0).abs().max().item()
    _log(f"{name}: shape={tuple(t.shape)}, nan={n_nan}, inf={n_inf}, max|x|={maxabs:.3e}")

def sanitize_(t, clamp=1e6):
    # In-place: replace NaN/Inf and clamp extremes (helps stabilize early)
    torch.nan_to_num_(t, nan=0.0, posinf=clamp, neginf=-clamp)
    t.clamp_(-clamp, clamp)
    return t

def masked_mse(pred, target, *, level_weights=None, high_level_threshold=2,
                 high_level_lambda=0.5, piecewise_threshold=4, piecewise_mae_lambda=0.5,
                 directional_lambda=0.3):
    """Weighted MSE with extra penalties for high congestion levels."""
    mask = torch.isfinite(target)
    if not mask.all():
        invalid = mask.numel() - mask.sum().item()
        _log(f"masked_mse: ignoring {invalid} invalid target elements")

    target_levels = torch.clamp(target.round(), 0, 5).long()

    if level_weights is None:
        base_weights = torch.tensor([0.25, 1.0, 2.0, 3.0, 4.0, 5.0], device=pred.device, dtype=pred.dtype)
    else:
        base_weights = level_weights.to(device=pred.device, dtype=pred.dtype)

    level_weights_expanded = base_weights[target_levels]
    level_weights_expanded = torch.where(mask, level_weights_expanded,
                                         torch.zeros((), device=pred.device, dtype=pred.dtype))

    diff2 = (pred - target) ** 2 * level_weights_expanded
    denom = level_weights_expanded.sum().clamp_min(1)
    mse_loss = diff2.sum() / denom

    total_loss = mse_loss

    if high_level_lambda > 0:
        high_mask = mask & (target_levels >= high_level_threshold)
        if high_mask.any():
            high_errors = (pred - target).abs()
            weighted_errors = high_errors * level_weights_expanded
            high_weight_sum = level_weights_expanded.where(high_mask,
                                                           torch.zeros_like(level_weights_expanded)).sum().clamp_min(1)
            mae_loss = weighted_errors.where(high_mask, torch.zeros_like(weighted_errors)).sum() / high_weight_sum
            total_loss = total_loss + high_level_lambda * mae_loss

    if piecewise_mae_lambda > 0:
        severe_mask = mask & (target >= piecewise_threshold)
        if severe_mask.any():
            severe_errors = (pred - target).abs()
            severe_weights = level_weights_expanded.where(severe_mask,
                                                          torch.zeros_like(level_weights_expanded))
            severe_weight_sum = severe_weights.sum().clamp_min(1)
            severe_loss = (severe_errors * severe_weights).sum() / severe_weight_sum
            total_loss = total_loss + piecewise_mae_lambda * severe_loss

    if directional_lambda > 0:
        under_mask = mask & (pred < target)
        if under_mask.any():
            directional_penalty = (target - pred) * level_weights_expanded
            penalty_sum = directional_penalty.where(under_mask,
                                                    torch.zeros_like(directional_penalty)).sum()
            under_weight_sum = level_weights_expanded.where(under_mask,
                                                            torch.zeros_like(level_weights_expanded)).sum().clamp_min(1)
            total_loss = total_loss + directional_lambda * penalty_sum / under_weight_sum

    return total_loss
def _sync_if_cuda():
    # make timings accurate on GPU
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def assert_edge_index_ok(edge_index, N):
    assert edge_index.dim() == 2 and edge_index.size(0) == 2, "edge_index must be (2, E)"
    assert edge_index.numel() > 0, "edge_index is empty"
    assert int(edge_index.min()) >= 0, "edge_index has negative node ids"
    assert int(edge_index.max()) < int(N), f"edge_index has node id >= N ({int(edge_index.max())} >= {int(N)})"

def log_mem(tag=""):
    if not use_cuda: return
    alloc = torch.cuda.memory_allocated() / (1024**2)
    reserv = torch.cuda.memory_reserved() / (1024**2)
    peak  = torch.cuda.max_memory_allocated() / (1024**2)
    print(f"[MEM] {tag} alloc={alloc:.0f}MB reserved={reserv:.0f}MB peak={peak:.0f}MB")

In [ ]:
num_epochs = num_epochs  # keep your value
stage_totals = defaultdict(float)

In [ ]:
# wrap it
picker = EpochUniquePicker(base_picker, p_top=0.5, on_exhaust="reset")

In [ ]:
# batch.keys()

In [ ]:
# x_dyn_past.shape

In [ ]:
torch.autograd.set_detect_anomaly(True)
grad_clip_norm = 1.0  # set to None to disable
stage_totals = defaultdict(float)
scaler = GradScaler(enabled=True)
for epoch in range(num_epochs):
    picker.start_epoch() 
    epoch_start = time.perf_counter()
    total_loss = 0.0
    prev_end = epoch_start

    pbar = tqdm(enumerate(traffic_dataloader), total=len(traffic_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")
    for step, batch in pbar:
        iter_start = time.perf_counter()
        data_wait_s = iter_start - prev_end
        stage_totals["data_wait"] += data_wait_s
        _log(f"Batch {step}: dataloader wait = {data_wait_s*1000:.1f} ms")

        # -------- move to device --------
        t0 = time.perf_counter()
        x_static         = batch["x_static"].to(device)              # (B,N,Fs)
        x_dyn_past       = batch["x_dynamic_past"].to(device)        # (B,N,Tin,Fd)
        x_dyn_future     = batch["x_dynamic_future"].to(device)      # (B,N,H,Ff)
        y_target         = batch["y"].to(device)                     # (B,N,H)
        edge_index       = batch["edge_index"].to(device)            # (2,E)
        _sync_if_cuda()
        t1 = time.perf_counter()
        move_s = t1 - t0
        stage_totals["to_device"] += move_s
        _log(f"Batch {step}: moved to device in {move_s*1000:.1f} ms")

        # -------- graph sizes + edge checks --------
        B = x_static.shape[0]
        N = x_static.shape[1]
        E = edge_index.shape[1]
        _log(f"Batch {step}: graph size -> B={B}, N={N}, E={E}")
        assert_edge_index_ok(edge_index, N)

        # -------- diagnostics BEFORE compute --------
        stats("x_static",         x_static)
        stats("x_dyn_past",       x_dyn_past)
        stats("x_dyn_future",     x_dyn_future)
        stats("y_target",         y_target)

        # sanitize inputs in-place (targets NOT sanitized; we mask them in loss)
        sanitize_(x_static)
        sanitize_(x_dyn_past)
        sanitize_(x_dyn_future)
        # optional static normalization:
        # normalize_static_(x_static)

        # -------- reshape/permute --------
        t0 = time.perf_counter()
        static_inputs = x_static[0]                                   # (N,Fs), shared across batch
        dynamic_seq_inputs   = x_dyn_past.permute(0, 2, 1, 3).contiguous()    # (B,Tin,N,Fd)
        future_known_inputs  = x_dyn_future.permute(0, 2, 1, 3).contiguous()  # (B,H,N,Ff)
        _sync_if_cuda()
        t1 = time.perf_counter()
        reshape_s = t1 - t0
        stage_totals["reshape"] += reshape_s
        _log(f"Batch {step}: reshape/permutation {reshape_s*1000:.1f} ms")

        # assert finiteness on model inputs
        assert torch.isfinite(static_inputs).all(),         "NaN/Inf in static_inputs"
        assert torch.isfinite(dynamic_seq_inputs).all(),    "NaN/Inf in dynamic_seq_inputs"
        assert torch.isfinite(future_known_inputs).all(),   "NaN/Inf in future_known_inputs"

        # -------- forward --------
        t0 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        preds = model(edge_index, static_inputs, dynamic_seq_inputs, future_known_inputs)  # (B,N,H)
        _sync_if_cuda()
        t1 = time.perf_counter()
        fwd_s = t1 - t0
        stage_totals["forward"] += fwd_s
        _log(f"Batch {step}: forward {fwd_s*1000:.1f} ms")

        # sanity check predictions
        if not torch.isfinite(preds).all():
            stats("preds (BAD)", preds)
            raise RuntimeError("Model produced NaN/Inf predictions")

        # -------- loss (masked) --------
        t0 = time.perf_counter()
        weights = torch.tensor([0.25, 1, 2, 3, 4, 5], device=preds.device, dtype=preds.dtype)
        loss = masked_mse(
            preds,
            y_target,
            level_weights=weights,
            high_level_threshold=2,
            high_level_lambda=0.5,       # optional override
            piecewise_threshold=3,
            piecewise_mae_lambda=0.5,
            directional_lambda=0.3,
        )
        _sync_if_cuda()
        t1 = time.perf_counter()
        loss_s = t1 - t0
        stage_totals["loss"] += loss_s
        _log(f"Batch {step}: loss {loss_s*1000:.1f} ms")

        if not torch.isfinite(loss):
            stats("preds", preds)
            stats("y_target", y_target)
            raise RuntimeError("Loss is NaN/Inf after masked_mse")

        # -------- backward + step --------
        t0 = time.perf_counter()
        loss.backward()
        if grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        _sync_if_cuda()
        t1 = time.perf_counter()
        bwd_s = t1 - t0
        stage_totals["backward+step"] += bwd_s
        _log(f"Batch {step}: backward+step {bwd_s*1000:.1f} ms")

        total_loss += float(loss)

        # -------- seed for next batch --------
        t0 = time.perf_counter()
        next_seed = picker.pick()
        traffic_dataset.reasign_seed(str(next_seed))
        t1 = time.perf_counter()
        seed_s = t1 - t0
        stage_totals["seed_pick"] += seed_s
        _log(f"Batch {step}: seed pick {seed_s*1000:.1f} ms -> {next_seed}")

        # -------- batch end --------
        batch_total = time.perf_counter() - iter_start
        stage_totals["batch_total"] += batch_total
        _log(f"Batch {step}: TOTAL {batch_total*1000:.1f} ms")
        pbar.set_postfix(loss=float(loss), N=N, E=E,fwd_ms=f"{fwd_s*1000:.0f}", bwd_ms=f"{bwd_s*1000:.0f}")
        torch.cuda.synchronize()
        torch.cuda.empty_cache() 
        
        prev_end = time.perf_counter()

        pbar.set_postfix(loss=float(loss), N=N, E=E,
                         fwd_ms=f"{fwd_s*1000:.0f}", bwd_ms=f"{bwd_s*1000:.0f}")

        
    epoch_time = time.perf_counter() - epoch_start
    avg_loss = total_loss / len(traffic_dataloader)
    _log(f"Epoch {epoch+1}: loss={avg_loss:.6f} | epoch time={epoch_time:.2f}s")

In [ ]:
y_target

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any, Dict, Optional
import torch
from torch import nn
from torch_geometric.data import Data

In [ ]:
def save_model_state(model, model_kwargs,path) -> str:
    """
    Save only what's essential to reuse later:
      - weights (state_dict)
      - the kwargs to rebuild your model class

    You do NOT need to save the dataloader, dataset, or optimizer.
    """
    ckpt = {
        "state_dict": model.state_dict(),
        "model_kwargs": model_kwargs,
        "torch_version": torch.__version__,
    }
    path = str(path)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(ckpt, path)
    return path


In [ ]:
# ============== LOAD ==============

def load_model_state(model_class: type[nn.Module],
                     path: str | Path,
                     map_location: str | torch.device = "cpu") -> nn.Module:
    """
    Rebuild the model and load weights.
    - You must provide the same model class you trained.
    - No need to recreate the dataloader.
    """
    ckpt = torch.load(str(path), map_location=map_location)
    model = model_class(**ckpt["model_kwargs"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model


# ============== SCORING HELPERS ==============

def make_data_for_inference(*,
                            x_static: torch.Tensor,                 # [B, F_static]
                            x_dynamic_past: torch.Tensor,           # [B, T_in,  F_past]
                            x_dynamic_future: torch.Tensor,         # [B, T_out, F_future]
                            edge_index: torch.Tensor,               # [2, E] (long)
                            x_weather: torch.Tensor, # [B, T_in+T_out, F_weather]
                            ) -> Data:
    """
    Build a torch_geometric.data.Data object aligned with your dataset’s output.
    Targets (y) are not needed for inference.
    """
    data_dict = {
        "x_static": x_static,
        "x_dynamic_past": x_dynamic_past,
        "x_dynamic_future": x_dynamic_future,
        "edge_index": edge_index.long(),
    }
    if x_weather is not None:
        data_dict["x_weather"] = x_weather
    return Data(**data_dict)


@torch.no_grad()
def score(model: nn.Module,
          *,
          x_static: torch.Tensor,
          x_dynamic_past: torch.Tensor,
          x_dynamic_future: torch.Tensor,
          edge_index: torch.Tensor,
          x_weather: torch.Tensor,
          device: str | torch.device = "cpu") -> torch.Tensor:
    """
    Run a forward pass on a batch. Works whether your model expects:
      - a PyG Data object (common in your pipeline), OR
      - unpacked tensors (if your forward already accepts them).

    It tries Data-first; if that fails, it retries with unpacked tensors.
    """
    model = model.to(device)
    x_static = x_static.to(device)
    x_dynamic_past = x_dynamic_past.to(device)
    x_dynamic_future = x_dynamic_future.to(device)
    edge_index = edge_index.to(device)
    x_weather = x_weather.to(device) if x_weather is not None else None

    # Try passing a Data object (matches your TrafficDataset style)
    data = make_data_for_inference(
        x_static=x_static,
        x_dynamic_past=x_dynamic_past,
        x_dynamic_future=x_dynamic_future,
        edge_index=edge_index,
        x_weather=x_weather,
    )
    try:
        out = model(data)
        return out
    except TypeError:
        # If your model.forward takes unpacked tensors, call it that way
        if x_weather is not None:
            return model(x_static, x_dynamic_past, x_dynamic_future, edge_index, x_weather)
        else:
            return model(x_static, x_dynamic_past, x_dynamic_future, edge_index)

In [ ]:
model_kwargs = dict(
    edge_index = None,
    static_dim = traffic_dataset.num_static_features,          # e.g. static node attributes
    dynamic_dim = dynamic_dim,       # total dynamic past features
    future_dim = traffic_dataset.num_dynamic_future_features,  # features known for future
    input_seq_len = 150,
    forecast_horizon = 50,                       # number of predicted timesteps
    model_type="GConvGRU"
)

In [ ]:
dynamic_dim

In [ ]:
traffic_dataset.num_static_features

In [ ]:
traffic_dataset.num_dynamic_features

In [ ]:
traffic_dataset.num_dynamic_future_features

In [ ]:
save_model_state(model, model_kwargs, "artifacts/traffic_gnn_v2_vel_2026_01_12.pt")

In [ ]:
# --- Evaluation helpers (level/speed + per-horizon + per-hop) ---
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict

class TemporalSubsetWithNodes(Dataset):
    """Wrap TemporalSubset to expose global node indices in each Data sample."""
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        data = self.subset[idx]
        absolute_idx = self.subset.start_time_idx + idx
        _, node_indices = self.subset.dataset._get_time_and_node_indices(absolute_idx)
        data.node_indices = torch.as_tensor(node_indices, dtype=torch.long)
        return data

def _denorm_targets(preds, targets, dataset):
    if getattr(dataset, 'target_norm_constants', None) is None:
        return preds, targets
    norm = dataset.target_norm_constants
    level_scale = float(norm.get('mean_jam_level', 1.0))
    velocity_scale = float(norm.get('mean_jam_velocity', 1.0))
    if targets.ndim == 4 and targets.size(-1) == 2:
        scale = torch.tensor([level_scale, velocity_scale], device=targets.device)
        preds = preds * scale
        targets = targets * scale
    else:
        preds = preds * level_scale
        targets = targets * level_scale
    return preds, targets

def _init_horizon_stats(h):
    return {
        'abs_error': torch.zeros(h),
        'sq_error': torch.zeros(h),
        'count': torch.zeros(h),
    }

def _update_horizon_stats(stats, err_abs, err_sq, mask):
    # err_*: (B, N, H)
    stats['abs_error'] += err_abs.sum(dim=(0, 1)).cpu()
    stats['sq_error'] += err_sq.sum(dim=(0, 1)).cpu()
    stats['count'] += mask.sum(dim=(0, 1)).cpu()

def _init_level_stats(h, num_levels=6):
    return {
        'abs_error': torch.zeros(h, num_levels),
        'sq_error': torch.zeros(h, num_levels),
        'count': torch.zeros(h, num_levels),
    }

def _update_level_stats(level_stats, preds_level, targets_level, num_levels=6):
    # preds/targets: (B, N, H)
    mask = torch.isfinite(targets_level)
    rounded = targets_level.round().long()
    err_abs = (preds_level - targets_level).abs()
    err_sq = (preds_level - targets_level).pow(2)
    for h in range(targets_level.size(-1)):
        mask_h = mask[:, :, h]
        if mask_h.sum() == 0:
            continue
        levels = rounded[:, :, h][mask_h]
        err_abs_h = err_abs[:, :, h][mask_h]
        err_sq_h = err_sq[:, :, h][mask_h]
        for lvl in range(num_levels):
            lvl_mask = levels == lvl
            count = lvl_mask.sum()
            if count > 0:
                level_stats['count'][h, lvl] += count.cpu()
                level_stats['abs_error'][h, lvl] += err_abs_h[lvl_mask].sum().cpu()
                level_stats['sq_error'][h, lvl] += err_sq_h[lvl_mask].sum().cpu()

def _init_corr_stats():
    return {'sum_x': 0.0, 'sum_y': 0.0, 'sum_xy': 0.0, 'sum_x2': 0.0, 'sum_y2': 0.0, 'count': 0}

def _update_corr_stats(stats, x, y):
    # x/y: 1D tensors
    stats['sum_x'] += float(x.sum())
    stats['sum_y'] += float(y.sum())
    stats['sum_xy'] += float((x * y).sum())
    stats['sum_x2'] += float((x * x).sum())
    stats['sum_y2'] += float((y * y).sum())
    stats['count'] += int(x.numel())

def _corr_from_stats(stats):
    n = stats['count']
    if n == 0:
        return float('nan')
    num = n * stats['sum_xy'] - stats['sum_x'] * stats['sum_y']
    den_x = n * stats['sum_x2'] - stats['sum_x'] ** 2
    den_y = n * stats['sum_y2'] - stats['sum_y'] ** 2
    if den_x <= 0 or den_y <= 0:
        return float('nan')
    return num / (den_x ** 0.5 * den_y ** 0.5)


In [ ]:
# --- Evaluate on test split (1-2 epochs) ---
eval_epochs = 1  # set to 2 if you want
max_batches_eval = None  # set to an int to limit
num_levels = 6

test_eval_dataset = TemporalSubsetWithNodes(test_dataset)
eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=pad_collate,
)

graph = traffic_dataset.graph
id_to_node = traffic_dataset.id_to_node

# Accumulators
horizon_stats_level = None
horizon_stats_velocity = None
level_stats = None
hop_stats = defaultdict(lambda: {'abs_err_level': 0.0, 'sq_err_level': 0.0, 'cnt_level': 0, 'abs_err_vel': 0.0, 'sq_err_vel': 0.0, 'cnt_vel': 0})
corr_level = _init_corr_stats()
corr_velocity = _init_corr_stats()

model.eval()
with torch.no_grad():
    for eval_epoch in range(1, eval_epochs + 1):
        picker.start_epoch()
        current_seed = picker.pick()
        test_dataset.reasign_seed(current_seed)

        for step_idx, batch in tqdm(enumerate(eval_loader), desc=f"Eval epoch {eval_epoch}"):
            if max_batches_eval is not None and step_idx >= max_batches_eval:
                break

            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            static_inputs = batch['x_static'][0]
            dynamic_seq_inputs = batch['x_dynamic_past'].permute(0, 2, 1, 3).contiguous()
            if 'x_weather' in batch and batch['x_weather'] is not None:
                x_weather = batch['x_weather'].permute(0, 2, 1, 3).contiguous()
                dynamic_seq_inputs = torch.cat([dynamic_seq_inputs, x_weather], dim=-1)
            future_known_inputs = batch['x_dynamic_future'].permute(0, 2, 1, 3).contiguous()
            edge_index = batch['edge_index']
            targets = batch['y']

            preds = model(edge_index, static_inputs, dynamic_seq_inputs, future_known_inputs)
            preds, targets = _denorm_targets(preds, targets, traffic_dataset)

            if targets.ndim == 4 and targets.size(-1) == 2:
                preds_level = preds[..., 0]
                preds_velocity = preds[..., 1]
                targets_level = targets[..., 0]
                targets_velocity = targets[..., 1]
            else:
                preds_level = preds
                targets_level = targets
                preds_velocity = None
                targets_velocity = None

            # Init stats after first batch (get horizon length)
            if horizon_stats_level is None:
                H = targets_level.size(-1)
                horizon_stats_level = _init_horizon_stats(H)
                level_stats = _init_level_stats(H, num_levels=num_levels)
                if targets_velocity is not None:
                    horizon_stats_velocity = _init_horizon_stats(H)

            # Per-horizon stats (level)
            mask_level = torch.isfinite(targets_level)
            err_abs_level = (preds_level - targets_level).abs()
            err_sq_level = (preds_level - targets_level).pow(2)
            _update_horizon_stats(horizon_stats_level, err_abs_level, err_sq_level, mask_level)
            _update_level_stats(level_stats, preds_level, targets_level, num_levels=num_levels)

            # Per-horizon stats (velocity)
            if targets_velocity is not None:
                mask_vel = torch.isfinite(targets_velocity)
                err_abs_vel = (preds_velocity - targets_velocity).abs()
                err_sq_vel = (preds_velocity - targets_velocity).pow(2)
                _update_horizon_stats(horizon_stats_velocity, err_abs_vel, err_sq_vel, mask_vel)

            # Correlation (level / velocity)
            _update_corr_stats(corr_level, preds_level[mask_level], targets_level[mask_level])
            if targets_velocity is not None:
                _update_corr_stats(corr_velocity, preds_velocity[mask_vel], targets_velocity[mask_vel])

            # Per-hop stats (center = current_seed from picker)
            node_indices = batch.get('node_indices', None)
            if node_indices is not None:
                seed_node = current_seed
                try:
                    dist_map = nx.single_source_shortest_path_length(graph, seed_node)
                except Exception:
                    dist_map = {}

                # Aggregate per-node across batch+time
                node_indices = node_indices.tolist()
                level_abs_per_node = err_abs_level.sum(dim=(0, 2)).cpu().numpy()
                level_sq_per_node = err_sq_level.sum(dim=(0, 2)).cpu().numpy()
                level_cnt_per_node = mask_level.sum(dim=(0, 2)).cpu().numpy()
                if targets_velocity is not None:
                    vel_abs_per_node = err_abs_vel.sum(dim=(0, 2)).cpu().numpy()
                    vel_sq_per_node = err_sq_vel.sum(dim=(0, 2)).cpu().numpy()
                    vel_cnt_per_node = mask_vel.sum(dim=(0, 2)).cpu().numpy()
                else:
                    vel_abs_per_node = vel_sq_per_node = vel_cnt_per_node = None

                for i, node_idx in enumerate(node_indices):
                    node_label = id_to_node[node_idx]
                    hop = dist_map.get(node_label, None)
                    if hop is None:
                        continue
                    stats = hop_stats[int(hop)]
                    stats['abs_err_level'] += float(level_abs_per_node[i])
                    stats['sq_err_level'] += float(level_sq_per_node[i])
                    stats['cnt_level'] += int(level_cnt_per_node[i])
                    if vel_abs_per_node is not None:
                        stats['abs_err_vel'] += float(vel_abs_per_node[i])
                        stats['sq_err_vel'] += float(vel_sq_per_node[i])
                        stats['cnt_vel'] += int(vel_cnt_per_node[i])

            # Prepare next batch seed
            current_seed = picker.pick()
            test_dataset.reasign_seed(current_seed)

# ---- Summaries ----
def _finalize_horizon(stats):
    mae = stats['abs_error'] / stats['count'].clamp(min=1)
    rmse = (stats['sq_error'] / stats['count'].clamp(min=1)).sqrt()
    return mae.numpy(), rmse.numpy()

mae_level, rmse_level = _finalize_horizon(horizon_stats_level)
horizon_df = pd.DataFrame({
    'horizon': np.arange(len(mae_level)),
    'mae_level': mae_level,
    'rmse_level': rmse_level,
})
if horizon_stats_velocity is not None:
    mae_vel, rmse_vel = _finalize_horizon(horizon_stats_velocity)
    horizon_df['mae_velocity'] = mae_vel
    horizon_df['rmse_velocity'] = rmse_vel

# Per-level (jam level 0..5) by horizon
level_mae = level_stats['abs_error'] / level_stats['count'].clamp(min=1)
level_rmse = (level_stats['sq_error'] / level_stats['count'].clamp(min=1)).sqrt()
level_mae_df = pd.DataFrame(level_mae.numpy(), columns=[f'level_{i}' for i in range(num_levels)])
level_mae_df.insert(0, 'horizon', np.arange(level_mae_df.shape[0]))
level_rmse_df = pd.DataFrame(level_rmse.numpy(), columns=[f'level_{i}' for i in range(num_levels)])
level_rmse_df.insert(0, 'horizon', np.arange(level_rmse_df.shape[0]))

# Per-hop summaries
hop_rows = []
for hop in sorted(hop_stats.keys()):
    s = hop_stats[hop]
    hop_rows.append({
        'hop': hop,
        'mae_level': s['abs_err_level'] / max(s['cnt_level'], 1),
        'rmse_level': (s['sq_err_level'] / max(s['cnt_level'], 1)) ** 0.5,
        'mae_velocity': s['abs_err_vel'] / max(s['cnt_vel'], 1) if s['cnt_vel'] else np.nan,
        'rmse_velocity': (s['sq_err_vel'] / max(s['cnt_vel'], 1)) ** 0.5 if s['cnt_vel'] else np.nan,
        'count': s['cnt_level'],
    })
hop_df = pd.DataFrame(hop_rows)

print('Pearson corr (level):', _corr_from_stats(corr_level))
if horizon_stats_velocity is not None:
    print('Pearson corr (velocity):', _corr_from_stats(corr_velocity))

display(horizon_df.head())
display(level_mae_df.head())
display(hop_df.head())
